In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
import accelerate
import torch

In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
import accelerate
import torch
import pandas as pd
import numpy as np
os.environ["HUGGINGFACE_TOKEN"] = "INSERT"
cache_dir='  '
model_name="CohereLabs/aya-expanse-8b"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=os.getenv("HUGGINGFACE_TOKEN"),cache_dir=cache_dir)
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.bfloat16, device_map="auto", use_auth_token=os.getenv("HUGGINGFACE_TOKEN"),cache_dir=cache_dir)




/home/rag24/.conda/envs/readability/lib/python3.11/site-packages/transformers/models/auto/tokenization_auto.py:823: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
/home/rag24/.conda/envs/readability/lib/python3.11/site-packages/transformers/models/auto/auto_factory.py:471: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [12]:
import pandas as pd
import numpy as np
file_path="greekhistoryv2.csv"
df=pd.read_csv(file_path)
df=df.drop(["Unnamed: 0"],axis=1)
#df=df.iloc[:,:5]
df.head()
#df.columns

,index,ID,text,course,level,AYA_32B_vanilla,AYA_32B_avg,AYA_32B_entropy,AYA_8B_vanilla,AYA_8B_avg,...,cefr_AYA_8B_avg,cefr_AYA_8B_entropy,conf_aya_32B_vanilla,conf_aya_32B_avg,conf_aya_32B_entropy,conf_aya_32B_verbal_conf,conf_noscale_aya_32B_vanilla,conf_noscale_aya_32B_avg,conf_noscale_aya_32B_entropy,conf_noscale_aya_32B_verbal_conf
0,47,48,Κυριαρχία\n ενυπάρχει εις το Έθνος· πάσα εξουσ...,history,12,8,8.396373,0.052133,8,8.005240,...,4.10836726263733,0.033471,10,9.163709,0.076892,7.997198834435153,8,7.622701846516414,0.053368,7.268945751706269
1,13,14,Ένας νεολιθικός\n είναι κατασκευασμένα από κλα...,history,10,7,7.070649,0.028780,8,7.980358,...,3.0232639746682253,0.008970,8,7.925984,0.021669,7.006688947318253,7,6.999239618536116,0.000599,7.995780169701902
2,173,174,"Μανολιό μου,\n Σε λίγες μέρες θα ξημερώσει 14\...",history,4,5,5.037540,0.013129,4,4.070716,...,2.004043110882151,0.002151,6,5.777365,0.043101,7.798184883053636,4,4.481976865052275,0.067431,7.042041530978054
3,87,88,Ματιά στο παρελθόν\n Η πρώτη αναγνώριση της «α...,history,6,7,7.006539,0.006083,7,6.725749,...,3.004607847327051,0.002363,6,6.474254,0.057209,7.946595979775594,7,6.838120710450141,0.035431,7.992273688971181
4,133,134,πολέμου. Σε αυτό συνέβαλε και η ίδρυση στη Μόσ...,history,12,8,8.032982,0.011375,8,7.967346,...,3.9635786561491386,0.013354,8,8.017251,0.006764,7.008569844754675,7,7.000898494953404,0.000597,7.99896587904081


In [11]:
## AYA for grades 2-6 - Language
corrs=[]   
scores_llm_avg=[]
scores_llm_vanilla=[]
real_scores=[]
entropies=[]
import pandas as pd
import numpy as np
for i in range(len(df)):
    count=0
    count1=0
    s=df.iloc[i,2]
    score=df.iloc[i,4]
    #Original prompt - grade scale
    #prompt=f"Rate the readability of an excerpt of a Greek language textbook between 2 (very easy to understand) and 6 (moderately challenging to understand) using the following scale: Considering the factors such as sentence structure, vocabulary and grammar complexity, and overall clarity, which grade level of Greek students is the text the most suitable for? For example, the answer would be 4 if the text is most suitable for Greek students in fourth grade or older. You may use both the provided scale and your own understanding of readability to determine the most appropriate score. Where on the scale of readability does this text rate: '{s}'. Only answer in this format: Answer: [SCORE] Explanation: [EXPLANATION]"
    #No scale prompt (0-9)
    prompt=f"Rate the readability of excerpts from Greek language textbooks targeted for students between second and sixth grades. Rate the readability of each passage with a whole number value between 1 (very easy to understand) and 9 (very difficult to understand). Consider factors such as sentence structure, vocabulary or grammar complexity, and overall clarity, as well as your own understanding of readability.  Where on the scale of readability does this text rate: '{s}'. Answer with this format: Answer: [SCORE] Explanation: [EXPLANATION]"
    ## CEFR scale prompt
    #prompt=f"Rate the readability of excerpts from Greek language textbooks targeted for students between second and sixth grades. Rate the readability of the text between 1 (very easy) and 6 (very challenging) using the following scale: 1 = Can understand very short, simple texts a single phrase at a time, picking up familiar names, words and basic phrases and rereading as required. 2 = Can understand short, simple texts on familiar matters of a concrete type 3 = Can read straightforward factual texts on subjects related to his/her field and interest with a satisfactory level of comprehension. 4 = Can read with a large degree of independence, adapting style and speed of reading to different texts and purpose 5 = Can understand in detail lengthy, complex texts, whether or not they relate to his/her own area of speciality, provided he/she can reread difficult sections. 6 = Can understand and interpret critically virtually all forms of the written language including abstract, structurally complex, or highly colloquial literary and non-literary writings. You may use both the provided scale and your own understanding of readability to determine the most appropriate score. Where on the scale of readability does this text rate: '{s}'. Answer with this format: Answer: [SCORE] Explanation: [EXPLANATION]"
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",return_dict=True).to("cuda")
    #inputs = tokenizer(prompt, return_tensors="pt").to("cuda")  # Move to GPU if available
    #print(i)
    #for j in range(5):
    outputs = model.generate(**input_ids,max_new_tokens=4,return_dict_in_generate=True,output_scores=True,pad_token_id=tokenizer.eos_token_id)
    scores = outputs.scores  
    #second to last token from generating 5 will have number, so change to 4 and get last num
    logits = scores[-1][0] 
    import torch
    probs = torch.softmax(logits, dim=-1)
    entropy = -torch.sum(probs * torch.log(probs + 1e-12)).item()
    vocab_size = probs.shape[0]
    normalized_entropy = entropy / torch.log(torch.tensor(vocab_size, dtype=torch.float)).item()
    #six possible number values so we take top6...it will not always consider all 6. Some have 0.0 probability
    top6_probs, top6_ids = torch.topk(probs, 6)
    top6_tokens=tokenizer.convert_ids_to_tokens(top6_ids.tolist())
    avg=0
    flag=False
    try:
        vanilla=int(top6_tokens[0])
        scores_llm_vanilla.append(vanilla)
    except ValueError:
        print('ERROR - FIRST TOKEN NOT NUM',top6_tokens)
        scores_llm_vanilla.append('na')
        scores_llm_avg.append('na')
        flag=True
    if flag==False:
        for (n,p) in zip(top6_tokens,top6_probs.tolist()):
            try:
                avg+=int(n)*p
            except ValueError:
                #print(avg)
                break
        scores_llm_avg.append(avg)
    real_scores.append(score)
    entropies.append(normalized_entropy)
        #try:
        #    count+=int(tokenizer.decode(outputs[0][-1]))
        #    count1+=1
        #except ValueError:
        #    try:
            #print(tokenizer.decode(outputs[0][-4:]),int(tokenizer.decode(outputs[0][-1])))
        #        count+=int(tokenizer.decode(outputs[0][-2]))
         #       count1+=1
         #   except ValueError:
         #       print('error')
         #       print(tokenizer.decode(outputs[0][-5:]))
   # try:
   #     scores_llm.append(count/count1)
   #     scores.append(score)
   # except ValueError:
   #     print('Big error')
   # except ZeroDivisionError:
   #     print('Zero error')
   #     scores_llm.append('na')
   #     scores.append('na')

filtered_scores, filtered_scores_vanilla,filtered_scores_avg = zip(*[(s, llm_v,llm_a) for s, llm_v,llm_a in zip(real_scores, scores_llm_vanilla,scores_llm_avg) if s != 'na' and llm_v != 'na'])
print(np.corrcoef(filtered_scores, filtered_scores_vanilla)[0, 1],len(filtered_scores_vanilla),np.corrcoef(filtered_scores, filtered_scores_avg)[0, 1],len(filtered_scores_avg))
#corrs.append(np.corrcoef(filtered_scores, filtered_scores_llm)[0, 1])
#print(np.corrcoef(scores, scores_llm)[0, 1])
df['noscale_AYA_8B_vanilla']=scores_llm_vanilla
df['noscale_AYA_8B_avg']=scores_llm_avg
df['noscale_AYA_8B_entropy']=entropies
#print(corrs)
df.to_csv("greek_languagev2.csv")

ERROR - FIRST TOKEN NOT NUM ['Ġthe', 'Ġeach', 'Ġthese', 'Ġreadability', 'Ġand', 'Ġthis']
ERROR - FIRST TOKEN NOT NUM ['Ġ', 'Ġ[', 'Ġ(', '**', 'ĠText', 'Ġ**']
ERROR - FIRST TOKEN NOT NUM ['Ġreadability', 'Ġratings', 'Ġrated', 'Ġreadings', 'Ġestimated', 'Ġscores']
ERROR - FIRST TOKEN NOT NUM ['Ġthe', 'Ġeach', 'Ġthese', 'Ġthose', 'Ġyour', 'Ġcada']
ERROR - FIRST TOKEN NOT NUM ['Ġthe', 'Ġeach', 'Ġthese', 'Ġreadability', 'Ġand', 'Ġtheir']
ERROR - FIRST TOKEN NOT NUM ['Ġ', 'Ġ[', 'ĠText', 'Ġ**', 'Ġ(', '**']
0.17470153117883716 387 0.196485176588236 387


In [17]:
## AYA for grades 4-12 - History
corrs=[]   
scores_llm_avg=[]
scores_llm_vanilla=[]
real_scores=[]
entropies=[]
import pandas as pd
import numpy as np
for i in range(len(df)):
    count=0
    count1=0
    s=df.iloc[i,2]
    score=df.iloc[i,4]
    prompt=f"Rate the readability of an excerpt of a Greek history textbook between 4 (very easy to understand) and 12 (difficult to understand) using the following scale: Considering the factors such as: sentence structure, vocabulary and grammar complexity, and overall clarity, which grade level of Greek students is the text the most suitable for? For example, the answer would be 6 if the text is most suitable for Greek students in sixth grade or older. You may use both the provided scale and your own understanding of readability to determine the most appropriate score. Where on the scale of readability does this text rate: '{s}'. Only answer in this format: Answer: [SCORE] Explanation: [EXPLANATION]"
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",return_dict=True).to("cuda")
    #inputs = tokenizer(prompt, return_tensors="pt").to("cuda")  # Move to GPU if available
    #print(i)
    #for j in range(5):
    outputs = model.generate(**input_ids,max_new_tokens=5,return_dict_in_generate=True,output_scores=True,pad_token_id=tokenizer.eos_token_id)
    scores = outputs.scores  
    #second to last token from generating 5 will have number, so change to 4 and get last num
    logits = scores[-2][0] 
    import torch
    probs = torch.softmax(logits, dim=-1)
    entropy = -torch.sum(probs * torch.log(probs + 1e-12)).item()
    vocab_size = probs.shape[0]
    normalized_entropy = entropy / torch.log(torch.tensor(vocab_size, dtype=torch.float)).item()
    #six possible number values so we take top6...it will not always consider all 6. Some have 0.0 probability
    top6_probs, top6_ids = torch.topk(probs, 6)
    top6_tokens=tokenizer.convert_ids_to_tokens(top6_ids.tolist())
    avg=0
    flag=False
    try:
        vanilla=int(top6_tokens[0])
        if vanilla==1:
            logits2=scores[-1][0]
            probs2 = torch.softmax(logits2, dim=-1)
            top6_probs2, top6_ids2 = torch.topk(probs2, 6)
            top6_tokens2=tokenizer.convert_ids_to_tokens(top6_ids2.tolist())
            vanilla=int(top6_tokens2[0])+10
        scores_llm_vanilla.append(vanilla)
    except ValueError:
        print('ERROR - FIRST TOKEN NOT NUM',top6_tokens)
        scores_llm_vanilla.append('na')
        scores_llm_avg.append('na')
        flag=True
    if flag==False:
        if vanilla<10:
            for (n,p) in zip(top6_tokens,top6_probs.tolist()):
                try:
                    temp=int(n)
                    if temp==1:
                        temp=10
                    avg+=int(temp)*p
                except ValueError:
                    #print(avg)
                    break
        if vanilla>=10:
            print(top6_tokens,top6_tokens2,top6_probs.tolist(),top6_probs2.tolist())
            for (n,p) in zip(top6_tokens,top6_probs.tolist()):
                    try:
                        temp=int(n)
                        if temp==1:
                            temp_p=p
                            temp=0
                        avg+=int(temp)*p
                    except ValueError:
                        #print(avg)
                        break
            for (n,p) in zip(top6_tokens2,top6_probs2.tolist()):
                    try:
                        temp=int(n)+10
                        avg+=int(temp)*p*temp_p
                    except ValueError:
                        #print(avg)
                        break    
            print(vanilla,avg)
        scores_llm_avg.append(avg)
    real_scores.append(score)
    entropies.append(normalized_entropy)
        #try:
        #    count+=int(tokenizer.decode(outputs[0][-1]))
        #    count1+=1
        #except ValueError:
        #    try:
            #print(tokenizer.decode(outputs[0][-4:]),int(tokenizer.decode(outputs[0][-1])))
        #        count+=int(tokenizer.decode(outputs[0][-2]))
         #       count1+=1
         #   except ValueError:
         #       print('error')
         #       print(tokenizer.decode(outputs[0][-5:]))
   # try:
   #     scores_llm.append(count/count1)
   #     scores.append(score)
   # except ValueError:
   #     print('Big error')
   # except ZeroDivisionError:
   #     print('Zero error')
   #     scores_llm.append('na')
   #     scores.append('na')

filtered_scores, filtered_scores_vanilla,filtered_scores_avg = zip(*[(s, llm_v,llm_a) for s, llm_v,llm_a in zip(real_scores, scores_llm_vanilla,scores_llm_avg) if s != 'na' and llm_v != 'na'])
print(np.corrcoef(filtered_scores, filtered_scores_vanilla)[0, 1],len(filtered_scores_vanilla),np.corrcoef(filtered_scores, filtered_scores_avg)[0, 1],len(filtered_scores_avg))
#corrs.append(np.corrcoef(filtered_scores, filtered_scores_llm)[0, 1])
#print(np.corrcoef(scores, scores_llm)[0, 1])
df['AYA_8B_vanilla']=scores_llm_vanilla
df['AYA_8B_avg']=scores_llm_avg
df['AYA_8B_entropy']=entropies
#print(corrs)
df.to_csv("greekhistoryv2.csv")

['1', '8', '9', '7', '6', '4'] ['0', '2', '1', '8', '9', '4'] [0.7677283883094788, 0.15117467939853668, 0.08091797679662704, 0.0001770073431544006, 1.052524225997331e-06, 8.197067131732183e-07] [0.9998018145561218, 0.00012338533997535706, 7.48370002838783e-05, 1.818601158554145e-09, 1.4163279438150767e-09, 7.581057448469153e-10]
10 9.61643899937649
['1', '9', '8', '7', '6', '5'] ['0', '1', '2', '9', '8', '6'] [0.730613112449646, 0.2093241661787033, 0.0599723756313324, 9.016481635626405e-05, 1.9723472632904304e-07, 1.428763685140666e-08] [0.9999535083770752, 4.539782094070688e-05, 1.06765435248235e-06, 6.691275311609957e-10, 6.691275311609957e-10, 2.1723388365924023e-10]
10 9.67049459163399
['1', '9', '8', '7', '6', '5'] ['0', '1', '2', '9', '8', '4'] [0.4017377495765686, 0.35453230142593384, 0.24366626143455505, 6.365989247569814e-05, 2.575966995266299e-08, 4.205150450076189e-09] [0.9998194575309753, 0.0001795277785276994, 1.0675112207536586e-06, 8.590615441228522e-10, 5.21047149870668

In [13]:
## AYA for grades 4-12 - History. CEFR and NO scale
corrs=[]   
scores_llm_avg=[]
scores_llm_vanilla=[]
real_scores=[]
entropies=[]
import pandas as pd
import numpy as np
for i in range(len(df)):
    count=0
    count1=0
    s=df.iloc[i,2]
    score=df.iloc[i,4]
    #No scale prompt (0-9)
    prompt=f"Rate the readability of excerpts from Greek History textbooks targeted for students between fourth and twelfth grades. Rate the readability of each passage with a whole number value between 1 (very easy to understand) and 9 (very difficult to understand). Consider factors such as sentence structure, vocabulary or grammar complexity, and overall clarity, as well as your own understanding of readability.  Where on the scale of readability does this text rate: '{s}'. Answer with this format: Answer: [SCORE] Explanation: [EXPLANATION]"
    ## CEFR scale prompt
    #prompt=f"Rate the readability of excerpts from Greek History textbooks targeted for students between fourth and twelfth grades. Rate the readability of the text between 1 (very easy) and 6 (very challenging) using the following scale: 1 = Can understand very short, simple texts a single phrase at a time, picking up familiar names, words and basic phrases and rereading as required. 2 = Can understand short, simple texts on familiar matters of a concrete type 3 = Can read straightforward factual texts on subjects related to his/her field and interest with a satisfactory level of comprehension. 4 = Can read with a large degree of independence, adapting style and speed of reading to different texts and purpose 5 = Can understand in detail lengthy, complex texts, whether or not they relate to his/her own area of speciality, provided he/she can reread difficult sections. 6 = Can understand and interpret critically virtually all forms of the written language including abstract, structurally complex, or highly colloquial literary and non-literary writings. You may use both the provided scale and your own understanding of readability to determine the most appropriate score. Where on the scale of readability does this text rate: '{s}'. Answer with this format: Answer: [SCORE] Explanation: [EXPLANATION]"
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",return_dict=True).to("cuda")
    #inputs = tokenizer(prompt, return_tensors="pt").to("cuda")  # Move to GPU if available
    #print(i)
    #for j in range(5):
    outputs = model.generate(**input_ids,max_new_tokens=4,return_dict_in_generate=True,output_scores=True,pad_token_id=tokenizer.eos_token_id)
    scores = outputs.scores  
    #second to last token from generating 5 will have number, so change to 4 and get last num
    logits = scores[-1][0] 
    import torch
    probs = torch.softmax(logits, dim=-1)
    entropy = -torch.sum(probs * torch.log(probs + 1e-12)).item()
    vocab_size = probs.shape[0]
    normalized_entropy = entropy / torch.log(torch.tensor(vocab_size, dtype=torch.float)).item()
    #six possible number values so we take top6...it will not always consider all 6. Some have 0.0 probability
    top6_probs, top6_ids = torch.topk(probs, 6)
    top6_tokens=tokenizer.convert_ids_to_tokens(top6_ids.tolist())
    avg=0
    flag=False
    try:
        vanilla=int(top6_tokens[0])
        scores_llm_vanilla.append(vanilla)
    except ValueError:
        print('ERROR - FIRST TOKEN NOT NUM',top6_tokens)
        scores_llm_vanilla.append('na')
        scores_llm_avg.append('na')
        flag=True
    if flag==False:
        for (n,p) in zip(top6_tokens,top6_probs.tolist()):
            try:
                avg+=int(n)*p
            except ValueError:
                #print(avg)
                break
        scores_llm_avg.append(avg)
    real_scores.append(score)
    entropies.append(normalized_entropy)
        #try:
        #    count+=int(tokenizer.decode(outputs[0][-1]))
        #    count1+=1
        #except ValueError:
        #    try:
            #print(tokenizer.decode(outputs[0][-4:]),int(tokenizer.decode(outputs[0][-1])))
        #        count+=int(tokenizer.decode(outputs[0][-2]))
         #       count1+=1
         #   except ValueError:
         #       print('error')
         #       print(tokenizer.decode(outputs[0][-5:]))
   # try:
   #     scores_llm.append(count/count1)
   #     scores.append(score)
   # except ValueError:
   #     print('Big error')
   # except ZeroDivisionError:
   #     print('Zero error')
   #     scores_llm.append('na')
   #     scores.append('na')

filtered_scores, filtered_scores_vanilla,filtered_scores_avg = zip(*[(s, llm_v,llm_a) for s, llm_v,llm_a in zip(real_scores, scores_llm_vanilla,scores_llm_avg) if s != 'na' and llm_v != 'na'])
print(np.corrcoef(filtered_scores, filtered_scores_vanilla)[0, 1],len(filtered_scores_vanilla),np.corrcoef(filtered_scores, filtered_scores_avg)[0, 1],len(filtered_scores_avg))
#corrs.append(np.corrcoef(filtered_scores, filtered_scores_llm)[0, 1])
#print(np.corrcoef(scores, scores_llm)[0, 1])
df['noscale_AYA_8B_vanilla']=scores_llm_vanilla
df['noscale_AYA_8B_avg']=scores_llm_avg
df['noscale_AYA_8B_entropy']=entropies
#print(corrs)
df.to_csv("greekhistoryv2.csv")

ERROR - FIRST TOKEN NOT NUM ['Ġthe', 'Ġeach', 'Ġthese', 'Ġreadability', 'Ġtheir', 'ĠEach']
ERROR - FIRST TOKEN NOT NUM ['Ġthe', 'Ġeach', 'Ġthese', 'Ġreadability', 'Ġtheir', 'Ġthis']
ERROR - FIRST TOKEN NOT NUM ['Ġthe', 'Ġeach', 'Ġthese', 'Ġcada', 'each', 'ĠEach']
ERROR - FIRST TOKEN NOT NUM ['Ġ', 'Ġ**', 'ĠĠ', 'Ġ[', '**', '3']
ERROR - FIRST TOKEN NOT NUM ['Ġthe', 'Ġeach', 'Ġthese', 'Ġreadability', 'Ġboth', 'Ġthis']
ERROR - FIRST TOKEN NOT NUM ['Ġ', 'Ġ**', 'Ġ[', 'ĠText', 'Ġ(', '**']
ERROR - FIRST TOKEN NOT NUM ['Ġreadability', 'Ġratings', 'Ġrated', 'Ġreadings', 'Ġestimated', 'Ġevaluations']
ERROR - FIRST TOKEN NOT NUM ['Ġ', 'Ġ**', 'Ġ[', '**', 'ĠText', 'ĠĠ']
ERROR - FIRST TOKEN NOT NUM ['Ġthe', 'Ġeach', 'Ġthese', 'Ġreadability', 'Ġand', 'Ġtheir']
ERROR - FIRST TOKEN NOT NUM ['Ġthe', 'Ġeach', 'Ġthese', 'Ġreadability', 'Ġa', 'Ġboth']
ERROR - FIRST TOKEN NOT NUM ['Ġeach', 'Ġthe', 'Ġthese', 'Ġreadability', 'Ġcada', 'each']
ERROR - FIRST TOKEN NOT NUM ['Ġ', 'Ġ**', 'ĠĠ', 'Ġ[', '**', '7']
ERROR 

In [9]:
## Confidence Greek history -- no scale and Greek language

corrs=[]   
scores_llm_avg=[]
scores_llm_vanilla=[]
real_scores=[]
entropies=[]
verbal_confs=[]
import pandas as pd
import numpy as np
for i in range(len(df)):
    count=0
    count1=0
    s=df.iloc[i,2]
    score=df.iloc[i,4]
    prompt=f"Rate the readability of an excerpt of a Greek language textbook between 2 (very easy to understand) and 6 (moderately challenging to understand) using the following scale: Considering the factors such as sentence structure, vocabulary and grammar complexity, and overall clarity, which grade level of Greek students is the text the most suitable for? For example, the answer would be 4 if the text is most suitable for Greek students in fourth grade or older. You may use both the provided scale and your own understanding of readability to determine the most appropriate score. Where on the scale of readability does this text rate: '{s}'. Additionally, state how confident you are that your rating will align with human raters, with a whole number value between 1 and 9. Only answer in this format: Answer: [SCORE] Confidence: [Confidence Score] Explanation: [EXPLANATION]"
    #prompt=f"Rate the readability of excerpts from Greek History textbooks targeted for students between fourth and twelfth grades. Rate the readability of the text with a whole number value between 1 (very easy to understand) and 9 (very difficult to understand). Consider factors such as sentence structure, vocabulary or grammar complexity, and overall clarity, as well as your own understanding of readability.  Where on the scale of readability does this text rate: '{s}'. Additionally, state how confident you are that your rating will align with human raters, with a whole number value between 1 and 9. Answer with this format: Answer: [SCORE] Confidence: [Confidence Score] Explanation: [EXPLANATION]"
    #prompt=f"Rate the readability of excerpts from Greek language textbooks targeted for students between second and sixth grades. Rate the readability of the text with a whole number value between 1 (very easy to understand) and 9 (very difficult to understand). Consider factors such as sentence structure, vocabulary or grammar complexity, and overall clarity, as well as your own understanding of readability.  Where on the scale of readability does this text rate: '{s}'. Additionally, state how confident you are that your rating will align with human raters, with a whole number value between 1 and 9. Answer with this format: Answer: [SCORE] Confidence: [Confidence Score] Explanation: [EXPLANATION]"
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",return_dict=True).to("cuda")
    #inputs = tokenizer(prompt, return_tensors="pt").to("cuda")  # Move to GPU if available
    #print(i)
    #for j in range(5):
    outputs = model.generate(**input_ids,max_new_tokens=10,return_dict_in_generate=True,output_scores=True,pad_token_id=tokenizer.eos_token_id)
    scores = outputs.scores  
    #second to last token from generating 5 will have number, so change to 4 and get last num
    logits_last=scores[-3][0]
    logits_fifth_to_last = scores[-7][0]
    import torch
    probs_last = torch.softmax(logits_last, dim=-1)
    probs_fifth_to_last = torch.softmax(logits_fifth_to_last, dim=-1)
    entropy = -torch.sum(probs_fifth_to_last * torch.log(probs_fifth_to_last + 1e-12)).item()
    vocab_size = probs_fifth_to_last.shape[0]
    normalized_entropy = entropy / torch.log(torch.tensor(vocab_size, dtype=torch.float)).item()
    #six possible number values so we take top6...it will not always consider all 6. Some have 0.0 probability
    top6_probs_last, top6_ids_last = torch.topk(probs_last, 6)
    top6_tokens_last=tokenizer.convert_ids_to_tokens(top6_ids_last.tolist())

    top6_probs_fifth_to_last, top6_ids_fifth_to_last = torch.topk(probs_fifth_to_last, 6)
    top6_tokens_fifth_to_last=tokenizer.convert_ids_to_tokens(top6_ids_fifth_to_last.tolist())

    
    avg=0
    flag=False
    try:
        vanilla=int(top6_tokens_fifth_to_last[0])
        scores_llm_vanilla.append(vanilla)
    except ValueError:
        print('ERROR - FIRST TOKEN NOT NUM')
        scores_llm_vanilla.append('na')
        scores_llm_avg.append('na')
        flag=True
    if flag==False:
        for (n,p) in zip(top6_tokens_fifth_to_last,top6_probs_fifth_to_last.tolist()):
            try:
                avg+=int(n)*p
            except ValueError:
                #print(avg)
                break
        scores_llm_avg.append(avg)
        
    avg2=0
    flag2=False
    try:
        vanilla_conf=int(top6_tokens_last[0])
    except ValueError:
        try:
            logits_last=scores[-2][0]
            probs_last = torch.softmax(logits_last, dim=-1)
            top6_probs_last, top6_ids_last = torch.topk(probs_last, 6)
            top6_tokens_last=tokenizer.convert_ids_to_tokens(top6_ids_last.tolist())
            vanilla_conf=int(top6_tokens_last[0])
        except ValueError:
            try:
                logits_last=scores[-1][0]
                probs_last = torch.softmax(logits_last, dim=-1)
                top6_probs_last, top6_ids_last = torch.topk(probs_last, 6)
                top6_tokens_last=tokenizer.convert_ids_to_tokens(top6_ids_last.tolist())
                vanilla_conf=int(top6_tokens_last[0])
            except ValueError:
                
                print('ERROR - FIRST CONFIDENCE TOKEN NOT NUM',tokenizer.decode(outputs['sequences'][0][-10:]))
                flag2=True
                verbal_confs.append('na')
    if flag2==False:
        for (n,p) in zip(top6_tokens_last,top6_probs_last.tolist()):
            try:
                avg2+=int(n)*p
            except ValueError:
                #print(avg)
                break
        verbal_confs.append(avg2)
    real_scores.append(score)
    entropies.append(normalized_entropy)

filtered_scores, filtered_scores_vanilla,filtered_scores_avg = zip(*[(s, llm_v,llm_a) for s, llm_v,llm_a in zip(real_scores, scores_llm_vanilla,scores_llm_avg) if s != 'na' and llm_v != 'na'])
print(np.corrcoef(filtered_scores, filtered_scores_vanilla)[0, 1],len(filtered_scores_vanilla),np.corrcoef(filtered_scores, filtered_scores_avg)[0, 1],len(filtered_scores_avg))
#corrs.append(np.corrcoef(filtered_scores, filtered_scores_llm)[0, 1])
#print(np.corrcoef(scores, scores_llm)[0, 1])
filtered_entropies, filtered_verbal_confs = zip(*[(e,v) for e,v in zip(entropies, verbal_confs) if e != 'na' and v != 'na'])
print(np.corrcoef(filtered_entropies, filtered_verbal_confs)[0, 1])

df['conf_aya_32B_vanilla']=scores_llm_vanilla
df['conf_aya_32B_avg']=scores_llm_avg
df['conf_aya_32B_entropy']=entropies
df['conf_aya_32B_verbal_conf']=verbal_confs
#print(corrs)
df.to_csv("greek_languagev2.csv")

0.4977720076371874 393 0.5071198855983283 393
-0.2053588648098205


In [3]:
## Confidence Greek history -- grade level

corrs=[]   
scores_llm_avg=[]
scores_llm_vanilla=[]
real_scores=[]
entropies=[]
verbal_confs=[]
import pandas as pd
import numpy as np
for i in range(len(df)):
    count=0
    count1=0
    s=df.iloc[i,2]
    score=df.iloc[i,4]
    prompt=f"Rate the readability of an excerpt of a Greek history textbook between 4 (very easy to understand) and 12 (difficult to understand) using the following scale: Considering the factors such as: sentence structure, vocabulary and grammar complexity, and overall clarity, which grade level of Greek students is the text the most suitable for? For example, the answer would be 6 if the text is most suitable for Greek students in sixth grade or older. You may use both the provided scale and your own understanding of readability to determine the most appropriate score. Where on the scale of readability does this text rate: '{s}'. Additionally, state how confident you are that your rating will align with human raters, with a whole number value between 1 and 9. Only answer in this format: Answer: [SCORE] Confidence: [Confidence Score] Explanation: [EXPLANATION]"
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",return_dict=True).to("cuda")
    #inputs = tokenizer(prompt, return_tensors="pt").to("cuda")  # Move to GPU if available
    #print(i)
    #for j in range(5):
    outputs = model.generate(**input_ids,max_new_tokens=11,return_dict_in_generate=True,output_scores=True,pad_token_id=tokenizer.eos_token_id)
    scores = outputs.scores  
    #second to last token from generating 5 will have number, so change to 4 and get last num
    logits_last=scores[-4][0]
    logits_fifth_to_last = scores[-8][0]
    import torch
    probs_last = torch.softmax(logits_last, dim=-1)
    probs_fifth_to_last = torch.softmax(logits_fifth_to_last, dim=-1)
    entropy = -torch.sum(probs_fifth_to_last * torch.log(probs_fifth_to_last + 1e-12)).item()
    vocab_size = probs_fifth_to_last.shape[0]
    normalized_entropy = entropy / torch.log(torch.tensor(vocab_size, dtype=torch.float)).item()
    #six possible number values so we take top6...it will not always consider all 6. Some have 0.0 probability
    top6_probs_last, top6_ids_last = torch.topk(probs_last, 6)
    top6_tokens_last=tokenizer.convert_ids_to_tokens(top6_ids_last.tolist())

    top6_probs_fifth_to_last, top6_ids_fifth_to_last = torch.topk(probs_fifth_to_last, 6)
    top6_tokens_fifth_to_last=tokenizer.convert_ids_to_tokens(top6_ids_fifth_to_last.tolist())

    
    #avg=0
    #flag=False
    #try:
    #    vanilla=int(top6_tokens_fifth_to_last[0])
    #    scores_llm_vanilla.append(vanilla)
    #except ValueError:
    #    print('ERROR - FIRST TOKEN NOT NUM')
    #    scores_llm_vanilla.append('na')
    #    scores_llm_avg.append('na')
    #    flag=True
    #if flag==False:
    #    for (n,p) in zip(top6_tokens_fifth_to_last,top6_probs_fifth_to_last.tolist()):
    #        try:
    #            avg+=int(n)*p
    #        except ValueError:
    #            #print(avg)
    #            break
    #    scores_llm_avg.append(avg)











    avg=0
    flag=False
    try:
        vanilla=int(top6_tokens_fifth_to_last[0])
        if vanilla==1:
            logits2=scores[-7][0]
            probs2 = torch.softmax(logits2, dim=-1)
            top6_probs2, top6_ids2 = torch.topk(probs2, 6)
            top6_tokens2=tokenizer.convert_ids_to_tokens(top6_ids2.tolist())
            vanilla=int(top6_tokens2[0])+10
        scores_llm_vanilla.append(vanilla)
    except ValueError:
        print('ERROR - FIRST TOKEN NOT NUM',top6_tokens)
        scores_llm_vanilla.append('na')
        scores_llm_avg.append('na')
        flag=True
    if flag==False:
        if vanilla<10:
            for (n,p) in zip(top6_tokens_fifth_to_last,top6_probs_fifth_to_last.tolist()):
                try:
                    temp=int(n)
                    if temp==1:
                        temp=10
                    avg+=int(temp)*p
                except ValueError:
                    #print(avg)
                    break
        if vanilla>=10:
            #print(top6_tokens_fifth_to_last,top6_tokens2,top6_probs_fifth_to_last.tolist(),top6_probs2.tolist())
            for (n,p) in zip(top6_tokens_fifth_to_last,top6_probs_fifth_to_last.tolist()):
                    try:
                        temp=int(n)
                        if temp==1:
                            temp_p=p
                            temp=0
                        avg+=int(temp)*p
                    except ValueError:
                        #print(avg)
                        break
            for (n,p) in zip(top6_tokens2,top6_probs2.tolist()):
                    try:
                        temp=int(n)+10
                        avg+=int(temp)*p*temp_p
                    except ValueError:
                        #print(avg)
                        break    
            #print(vanilla,avg)
        scores_llm_avg.append(avg)






        


    avg2=0
    flag2=False
    try:
        vanilla_conf=int(top6_tokens_last[0])
    except ValueError:
        try:
            logits_last=scores[-3][0]
            probs_last = torch.softmax(logits_last, dim=-1)
            top6_probs_last, top6_ids_last = torch.topk(probs_last, 6)
            top6_tokens_last=tokenizer.convert_ids_to_tokens(top6_ids_last.tolist())
            vanilla_conf=int(top6_tokens_last[0])
        except ValueError:
            try:
                logits_last=scores[-2][0]
                probs_last = torch.softmax(logits_last, dim=-1)
                top6_probs_last, top6_ids_last = torch.topk(probs_last, 6)
                top6_tokens_last=tokenizer.convert_ids_to_tokens(top6_ids_last.tolist())
                vanilla_conf=int(top6_tokens_last[0])
            except ValueError:
                try:
                    logits_last=scores[-1][0]
                    probs_last = torch.softmax(logits_last, dim=-1)
                    top6_probs_last, top6_ids_last = torch.topk(probs_last, 6)
                    top6_tokens_last=tokenizer.convert_ids_to_tokens(top6_ids_last.tolist())
                    vanilla_conf=int(top6_tokens_last[0])
                except ValueError:
                    
                    print('ERROR - FIRST CONFIDENCE TOKEN NOT NUM',tokenizer.decode(outputs['sequences'][0][-10:]))
                    flag2=True
                    verbal_confs.append('na')
    if flag2==False:
        for (n,p) in zip(top6_tokens_last,top6_probs_last.tolist()):
            try:
                avg2+=int(n)*p
            except ValueError:
                #print(avg)
                break
        verbal_confs.append(avg2)
    real_scores.append(score)
    entropies.append(normalized_entropy)

filtered_scores, filtered_scores_vanilla,filtered_scores_avg = zip(*[(s, llm_v,llm_a) for s, llm_v,llm_a in zip(real_scores, scores_llm_vanilla,scores_llm_avg) if s != 'na' and llm_v != 'na'])
print(np.corrcoef(filtered_scores, filtered_scores_vanilla)[0, 1],len(filtered_scores_vanilla),np.corrcoef(filtered_scores, filtered_scores_avg)[0, 1],len(filtered_scores_avg))
#corrs.append(np.corrcoef(filtered_scores, filtered_scores_llm)[0, 1])
#print(np.corrcoef(scores, scores_llm)[0, 1])
filtered_entropies, filtered_verbal_confs = zip(*[(e,v) for e,v in zip(entropies, verbal_confs) if e != 'na' and v != 'na'])
print(np.corrcoef(filtered_entropies, filtered_verbal_confs)[0, 1])

df['conf_aya_32B_vanilla']=scores_llm_vanilla
df['conf_aya_32B_avg']=scores_llm_avg
df['conf_aya_32B_entropy']=entropies
df['conf_aya_32B_verbal_conf']=verbal_confs
#print(corrs)
df.to_csv("greekhistoryv2.csv")

From v4.47 onwards, when a model cache is to be returned, `generate` will return a `Cache` instance instead by default (as opposed to the legacy tuple of tuples format). If you want to keep returning the legacy format, please set `return_legacy_cache=True`.


ERROR - FIRST CONFIDENCE TOKEN NOT NUM : 3-4

Confidence: 
0.4115818102940784 804 0.4434058559019119 804
0.22119366563320814
